# FormulaLM — PROMPT 7: Base Model Baseline Evaluation (Qwen3-1.7B, Non-Thinking)

Evaluates the **untouched** `Qwen/Qwen3-1.7B` base model (pinned revision `70d244cc86ccca08cf5af4e1e306ecf908b1ad5e`, non-thinking mode, `temperature=0`, `max_tokens=96`, `seed=42`) in 4 baseline settings:
1. `baseline_zeroshot_real` — 0-shot on `data/test_real/test_real_DRAFT.csv` (80 examples)
2. `baseline_5shot_real` — 5-shot (drawn from `train.jsonl`) on `data/test_real/test_real_DRAFT.csv` (80 examples)
3. `baseline_zeroshot_synthetic` — 0-shot on `data/processed/test_synthetic.jsonl` (500 examples)
4. `baseline_5shot_synthetic` — 5-shot (drawn from `train.jsonl`) on `data/processed/test_synthetic.jsonl` (500 examples)

**Important:** Connect to a **T4 GPU** runtime (not TPU — PyTorch `transformers` + Unsloth require NVIDIA CUDA, not TPU).

In [ ]:
# 1. Verify NVIDIA CUDA GPU (must NOT be TPU or CPU)
import torch

assert torch.cuda.is_available(), (
    'No CUDA GPU detected! Switch your Colab runtime/kernel from TPU/CPU to T4 GPU.'
)
print('GPU:', torch.cuda.get_device_name(0), '| VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
# 2. Clone the repo (works in both web Colab and VS Code / Antigravity Colab extension)
import getpass, os, pathlib, subprocess

REPO_DIR = pathlib.Path('/content/FormulaLM')
if not REPO_DIR.exists():
    token = os.environ.get('GITHUB_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            token = None
    if not token:
        # Private repo: paste a GitHub PAT when prompted (leave empty if repo is public)
        token = getpass.getpass('GitHub PAT for abubakarp789/FormulaLM (hidden, empty if public): ').strip()
    url = f'https://{token}@github.com/abubakarp789/FormulaLM.git' if token else 'https://github.com/abubakarp789/FormulaLM.git'
    subprocess.run(['git', 'clone', url, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

%cd /content/FormulaLM

In [ ]:
# 3. Install lightweight eval + transformers dependencies
%pip install -q formulas openpyxl pandas pyyaml requests tqdm accelerate "transformers>=4.51.0" "huggingface_hub>=0.30.0"
%pip install -q --no-deps -e .

In [ ]:
# 4. Regenerate verified train/val/test_synthetic JSONL splits (exact seed=42)
!python -m src.data.build_dataset --workers 2

In [ ]:
# 5. Load Qwen/Qwen3-1.7B ONCE on T4 GPU and run all 4 baseline evaluations
import sys
if '/content/FormulaLM/src' not in sys.path:
    sys.path.insert(0, '/content/FormulaLM/src')
from src.eval.run_eval import TransformersBackend, evaluate

MODEL_ID = 'Qwen/Qwen3-1.7B'
REVISION = '70d244cc86ccca08cf5af4e1e306ecf908b1ad5e'
SEED = 42
MAX_TOKENS = 96
BATCH_SIZE = 8

backend = TransformersBackend(
    MODEL_ID,
    revision=REVISION,
    load_in_4bit=False,
    enable_thinking=False,
)

runs = [
    ('baseline_zeroshot_real', 'data/test_real/test_real_DRAFT.csv', 0),
    ('baseline_5shot_real', 'data/test_real/test_real_DRAFT.csv', 5),
    ('baseline_zeroshot_synthetic', 'data/processed/test_synthetic.jsonl', 0),
    ('baseline_5shot_synthetic', 'data/processed/test_synthetic.jsonl', 5),
]

for run_name, dataset, few_shot in runs:
    print(f'=== Running {run_name} (few_shot={few_shot}) ===', flush=True)
    m = evaluate(
        backend=backend,
        dataset_path=dataset,
        run_name=run_name,
        seed=SEED,
        max_tokens=MAX_TOKENS,
        few_shot=few_shot,
        few_shot_source='data/processed/train.jsonl',
        batch_size=BATCH_SIZE,
        resume=True,
    )
    ov = m['overall']
    print(
        f"{run_name}: n={ov['n']} | syntax={100*ov['syntax_valid_rate']:.1f}% | "
        f"exec_acc={100*ov['execution_accuracy']:.1f}% | "
        f"exact={100*ov['exact_match_rate']:.1f}% | "
        f"raw_fmt={100*ov['raw_format_ok_rate']:.1f}% | "
        f"avg_lat={ov['avg_latency_ms']:.1f}ms",
        flush=True,
    )

In [ ]:
# 6. Produce baseline comparison table and top 15 failure patterns
!python -m src.eval.analyze_errors \
  --runs results/baseline_zeroshot_synthetic results/baseline_5shot_synthetic results/baseline_zeroshot_real results/baseline_5shot_real \
  --out results/baseline_summary.md \
  --json-out results/baseline_summary.json \
  --top-k 15

In [ ]:
# 7. Save results back to GitHub (works great from IDE Colab extension) OR zip for download
!zip -rq baseline_results.zip results/baseline_*
!ls -lh baseline_results.zip

# Set PUSH_TO_GITHUB = True if your PAT has Contents:Read+Write on abubakarp789/FormulaLM
PUSH_TO_GITHUB = True
if PUSH_TO_GITHUB:
    !git config user.name "abubakarp789"
    !git config user.email "abubakarp789@gmail.com"
    !git add results/baseline_*
    !git commit -m "results: add Qwen3-1.7B zero-shot and 5-shot baseline runs"
    !git pull --rebase origin main
    !git push origin main